In [4]:
%load_ext autoreload
%autoreload 2

import config
from dindex_store.discovery_index import load_dindex
from aurum_api.algebra import AurumAPI
from qbe_module.query_by_example import ExampleColumn, QueryByExample
from qbe_module.materializer import Materializer
from tqdm import tqdm
import os
import json
from view_distillation import vd
from view_distillation.vd import ViewDistillation
import time
from embedding_module.processor import Processor



The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
cnf = {setting: getattr(config, setting) for setting in dir(config)
       if setting.islower() and len(setting) > 2 and setting[:2] != "__"}

dindex = load_dindex(cnf)
print("Loading DIndex...OK")

api = AurumAPI(dindex)
print("created aurum api")

# QBE interface
qbe = QueryByExample(api)

Initializing profile index: duckdb...
Initializing content similarity index: simpleminhash...
Note that SimpleMHIndex is volatile and so it's not available after loading...
Initializing FTS index: duckdb...
Initializing Graph index: kuzu...
Loading DIndex...OK
created aurum api


In [26]:
"""
Specify an example query
"""
example_columns = [
    # ExampleColumn(attr='ethnicity', examples=["White", "Black"]),
    # ExampleColumn(attr='grade', examples=["5","6"]),
    # ExampleColumn(attr='number', examples=["1234561234", "1234567890"]),
    # ExampleColumn(attr='street', examples=["Main St", "Elm St"]),
    # ExampleColumn(attr='city', examples=["Bronx", "Brooklyn"]),
    ExampleColumn(attr='grade', examples=[]),
]

In [5]:

file_path = "nyc_open_data/"
processor = Processor(file_path)
sentences = processor.process_single()


100%|██████████| 1000/1000 [18:46<00:00,  1.13s/it] 


In [ ]:
from embedding_module.embedding_trainer import EmbeddingTrainer

trainer = EmbeddingTrainer(sentences)
trainer.train()
trainer.save_model("embedding_model.model")



In [27]:
from transform_module.transform import DateTransform

t = DateTransform(example_columns=example_columns)
queries = t.apply_transform()
for q in queries:
    print(q.attr)


grade


In [21]:
from gensim.models import KeyedVectors

word_vectors = KeyedVectors.load_word2vec_format('GoogleNews-vectors-negative300.bin', binary=True)

In [39]:
x = qbe.find_similar_columns(example_columns)
x

In [37]:
"""
Find candidate columns
"""
start = time.time()
candidate_list = qbe.find_candidate_columns(example_columns, cluster_prune=False)

# so basically finds all different columns that have value same as attr or similar

print("Time taken to find candidate columns: ", time.time() - start)

"""
Display candidate columns (for debugging purpose)
"""
for i, candidate in enumerate(candidate_list):
    print('column {}: found {} candidate columns'.format(format(i), len(candidate)))

    # for debugging purpose, print the candidate columns
    for col in candidate:
        print(col.to_str(), col.examples_set)

Time taken to find candidate columns:  0.028695106506347656
column 0: found 4 candidate columns
Chicago Public Schools - School Profile Information SY2223.csv.summary set()
Chicago Public Schools - School Profile Information SY2223.csv.secondary_contact_title set()
Chicago Public Schools - School Profile Information SY2223.csv.third_contact_title set()
Chicago Public Schools - School Profile Information SY2223.csv.fourth_contact_title set()
column 1: found 1 candidate columns
Chicago Public Schools - School Profile Information SY2223.csv.summary set()
column 2: found 1 candidate columns
Chicago Public Schools - School Profile Information SY2223.csv.summary set()


In [38]:
# CANDIDATE GROUPS
candidate_tbls = [set() for _ in candidate_list] 
cand_groups, tbl_cols = qbe.find_candidate_groups(candidate_list)
# takes in List[List[Column]]

print("number of candidate groups: ", len(cand_groups))

# """
# Find join graphs
# """
number_of_cand_groups_to_search = 200
join_graphs = qbe.find_join_graphs_for_cand_groups(cand_groups[0:number_of_cand_groups_to_search])
print(f"number of join graphs: {len(join_graphs)}")

number of candidate groups:  1


100%|██████████| 1/1 [00:00<00:00, 5652.70it/s]

number of join graphs: 1


In [30]:
data_path = './nyc_open_data/'  # path where the raw data is stored
output_path = './output/'  # path to store the output views
max_num_views = 200  # how many views you want to materialize
sep = ','  # csv separator

if not os.path.exists(output_path):
    os.makedirs(output_path)
materializer = Materializer(data_path, tbl_cols, 200, sep)

result_dfs = []

j = 0
for join_graph in tqdm(join_graphs):
    """
    a join graph can produce multiple views because different columns are projected
    """
    df_list = materializer.materialize_join_graph(join_graph)
    for df in df_list:
        if len(df) != 0:
            metadata = {}
            metadata["join_graph"] = join_graph.to_str()
            metadata["columns_proj"] = list(df.columns)
            with open(f"./{output_path}/view{j}.json", "w") as outfile:
                json.dump(metadata, outfile)
            j += 1
            print("non empty view", j)
            new_cols = []
            k = 1
            for col in df.columns:
                new_col = col.split(".")[-1]
                if new_col in new_cols:
                    new_col += str(k)
                    k += 1
                new_cols.append(new_col)
            df.columns = new_cols
            df.to_csv(f"./{output_path}/view{j}.csv", index=False)

            result_dfs.append(df)

    if j >= max_num_views:
        break

print("valid views", j)


0it [00:00, ?it/s]

valid views 0


In [ ]:
vd = ViewDistillation(dfs=result_dfs)

# Generates a networkx graph representing 4C relationships among views (nodes)
vd.generate_graph()

# Prune the graph with the given pruning strategies, returning the updated graph
graph = vd.prune_graph(remove_identical_views=True,
                       remove_contained_views=True,
                       union_complementary_views=True)



original num views: 1
num views after pruning compatible: 1
num views after pruning contained: 1
num views after union complementary: 1
num of contradictory view pairs: 0
